<a href="https://colab.research.google.com/github/marcocslima/rag_tests/blob/main/RAG_IA_Operacional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
caminho_pasta = "/content/drive/MyDrive/tmp/RAG/Documentos_Base"

#SETUP

In [ ]:
#!pip install -qU pymupdf4llm langchain langchain-community langchain-huggingface langchain-google-genai faiss-cpu sentence-transformers langchain-text-splitters
!pip install -qU pymupdf4llm langchain langchain-community langchain-huggingface langchain-google-genai faiss-cpu sentence-transformers langchain-text-splitters langchain-classic

In [ ]:
!pip install -qU gradio

In [ ]:
# Adicione ou substitua os imports por estes:
import os
import glob
from google.colab import userdata
from langchain_community.document_loaders import PyMuPDFLoader # NOVO LOADER
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
#from langchain.chains import create_history_aware_retriever, create_retrieval_chain # CRIA A MEMÓRIA
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
import gradio as gr

In [ ]:
# --- 1. CONFIGURAÇÃO DE AMBIENTE ---
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

# --- 2. INICIALIZAÇÃO DOS COMPONENTES GLOBAIS ---
# Variáveis que serão preenchidas pela função de carga de pasta
db = None
retriever = None
rag_chain = None

print("⚙️ Configurando componentes...")

# Divisor de Texto (Chunking)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=2500,     # Aumentamos de 1000 para 2500 caracteres
    chunk_overlap=400    # Aumentamos a sobreposição para não cortar ideias no meio
)

# Modelo de Embeddings (Local)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Modelo de Linguagem (Gemini 2.5 Flash)
llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview", #"gemini-2.5-flash",
    temperature=0.4
)

# --- 1. DEFINIÇÃO DO PROMPT ANALÍTICO ---
system_prompt = (
    "Você é um Consultor Sênior e Analista de Inteligência de Dados. Sua missão é fornecer respostas profundas, "
    "analíticas e altamente úteis baseadas estritamente nas premissas e informações do contexto fornecido.\n\n"
    "DIRETRIZES DE COMPORTAMENTO (Obrigatório):\n"
    "1. INTELIGÊNCIA ANALÍTICA: Não seja um mero buscador de palavras-chave. Leia nas entrelinhas. "
    "Você tem total liberdade para correlacionar dados, identificar padrões, fazer deduções lógicas e cruzar "
    "informações dentro do contexto para entregar uma resposta completa e inteligente.\n"
    "2. CONHECIMENTO EXTERNO COMO FERRAMENTA: Use sua inteligência e conhecimento geral APENAS de forma instrumental: "
    "para explicar jargões, estruturar raciocínios, realizar cálculos baseados nos dados fornecidos ou traduzir conceitos complexos. "
    "NUNCA use seu conhecimento externo para adicionar FATOS, DADOS ou EVENTOS que não existam no contexto.\n"
    "3. SÍNTESE SEM ALUCINAÇÃO: É permitido extrair consequências lógicas das premissas do texto (ex: se o texto diz que choveu muito, "
    "você pode deduzir que o solo ficou molhado). No entanto, é absolutamente proibido inventar informações para preencher lacunas.\n\n"
    "COMO LIDAR COM INFORMAÇÕES FALTANTES (Não trave):\n"
    "Se a pergunta pedir algo que não está explícito nem pode ser deduzido com segurança do contexto, não encerre a resposta abruptamente. "
    "Siga este protocolo:\n"
    "(A) Avise de forma transparente e educada que a informação exata não está disponível nos documentos.\n"
    "(B) Imediatamente, ofereça a resposta MAIS PRÓXIMA ou RELEVANTE usando as premissas que você tem. Mostre o que o documento *de fato* diz "
    "sobre o assunto abordado, ajudando o usuário a chegar a uma conclusão aproximada.\n\n"
    "Contexto dos Documentos:\n"
    "{context}" # <-- O Langchain vai injetar aqui automaticamente usando o create_stuff_documents_chain
)

# NOVO: Prompt para reformular perguntas baseado no histórico
contextualize_q_system_prompt = (
    "Dado um histórico de chat e a última pergunta do usuário, "
    "formule uma pergunta independente que possa ser compreendida sem o histórico. "
    "NÃO responda à pergunta, apenas reformule-a se necessário, senão retorne-a como está."
)
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# NOVO: Prompt principal com suporte a histórico
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# Função auxiliar para formatar os documentos recuperados
def formatar_docs(documentos):
    return "\n\n".join(doc.page_content for doc in documentos)

print("✅ Infraestrutura pronta! Agora execute a função de carregar a pasta.")

In [ ]:
def carregar_pasta(caminho_pasta):
    global db, retriever, rag_chain, splitter, embeddings, llm

    arquivos = glob.glob(os.path.join(caminho_pasta, "*.pdf"))
    if not arquivos:
        print(f"❌ Nenhum PDF encontrado em: {caminho_pasta}")
        return

    docs_list =[]
    print(f"🗂️ Processando {len(arquivos)} arquivos...")

    for f in arquivos:
        nome_f = os.path.basename(f)
        try:
            print(f"📖 Lendo: {nome_f}...", end=" ")

            # CORREÇÃO 1: Usando PyMuPDFLoader (Lê página por página sem travar)
            loader = PyMuPDFLoader(f)
            docs = loader.load()

            # Adiciona metadados e faz o split corretamente
            for doc in docs:
                doc.metadata["fonte"] = nome_f

            chunks = splitter.split_documents(docs)
            docs_list.extend(chunks)
            print("✅")
        except Exception as e:
            print(f"❌ Erro: {e}")

    if not docs_list: return

    # --- CRIAÇÃO DO RAG COM MEMÓRIA ---
    print(f"⚙️ Indexando {len(docs_list)} chunks no FAISS...")
    db = FAISS.from_documents(docs_list, embeddings)
    retriever = db.as_retriever(search_kwargs={"k": 60})

    # Cria o retriever que avalia o histórico antes de buscar
    history_aware_retriever = create_history_aware_retriever(
        llm, retriever, contextualize_q_prompt
    )

    # Cria a chain que insere os documentos no prompt
    question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

    # CORREÇÃO 2: A nova rag_chain robusta
    rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

    print("🚀 SISTEMA PRONTO PARA O CHAT!")

carregar_pasta(caminho_pasta)

In [ ]:
# --- Célula para Adicionar Novo Documento ---

# novo_pdf = "/content/declaracao-de-conclusao-da-obra.pdf" # <-- Coloque o caminho do novo arquivo aqui

# if os.path.exists(novo_pdf):
#     print(f"📄 Processando novo documento: {novo_pdf}")

#     # 1. Extrair texto do novo PDF
#     novo_texto_md = pymupdf4llm.to_markdown(novo_pdf)

#     # 2. Dividir em chunks (usando o mesmo splitter da Célula 3)
#     novos_docs = splitter.create_documents([novo_texto_md])
#     print(f"➕ Gerados {len(novos_docs)} novos chunks.")

#     # 3. ADICIONAR ao banco de dados FAISS existente
#     db.add_documents(novos_docs)

#     # 4. Atualizar o retriever para garantir que ele veja os novos dados
#     # Mantemos o k=15 para uma análise profunda
#     retriever = db.as_retriever(search_kwargs={"k": 15})

#     print("✅ Documento adicionado com sucesso ao conhecimento da IA!")
#     print(f"O banco de dados agora possui informações de múltiplos documentos.")
# else:
#     print("❌ Arquivo não encontrado. Verifique se o upload foi feito corretamente.")

#SERVIDOR WEB

In [ ]:
# Célula SERVIDOR WEB
import gradio as gr

def responder_chat(mensagem, historico):
    try:
        chat_history =[]
        # Novo loop blindado: Funciona em Gradio antigo e novo
        for msg in historico:
            # Se o Gradio enviar no formato moderno (Dicionários/Mensagens)
            if isinstance(msg, dict):
                role = "human" if msg.get("role") == "user" else "ai"
                chat_history.append((role, msg.get("content")))
            # Se o Gradio enviar como Objeto dataclass
            elif hasattr(msg, "role") and hasattr(msg, "content"):
                role = "human" if msg.role == "user" else "ai"
                chat_history.append((role, msg.content))
            # Se o Gradio enviar no formato clássico (Tuplas/Listas)
            elif isinstance(msg, (list, tuple)) and len(msg) == 2:
                chat_history.append(("human", msg[0]))
                chat_history.append(("ai", msg[1]))

        # Invocamos passando a mensagem atual e todo o histórico formatado!
        resposta = rag_chain.invoke({
            "input": mensagem,
            "chat_history": chat_history
        })

        return resposta["answer"]

    except Exception as e:
        return f"❌ Erro ao processar consulta: {e}"

demo = gr.ChatInterface(
    fn=responder_chat,
    title="🤖 Chat - RAG Analítico",
    description="IA Analítica treinada com os documentos. Ela lembrará do contexto da conversa!"
)

print("⏳ Gerando link de acesso...")
demo.launch(share=True, debug=True)

#CHAT

In [ ]:
# 4ª Célula: Chat Interativo com o PDF
from IPython.display import clear_output

clear_output(wait=True)
print("🤖 Chat Iniciado! Pergunte qualquer coisa sobre o documento.")
print("👉 (Para encerrar, digite 'sair', 'fim' ou 'exit')")
print("-" * 50)

while True:
    # Recebe a pergunta do usuário
    pergunta = input("\n👤 Você: ")

    # Condição para parar o chat
    if pergunta.lower().strip() in ['sair', 'fim', 'exit', 'parar']:
        print("\n👋 Chat encerrado. Até logo!")
        break

    # Ignora se o usuário apertar Enter sem digitar nada
    if not pergunta.strip():
        continue

    print("⏳ Assistente pesquisando no documento...", end="\r")

    try:
        # Envia a pergunta para o nosso RAG (LCEL)
        resposta = rag_chain.invoke(pergunta)

        # Imprime a resposta
        print(" " * 50, end="\r") # Limpa a linha do "pesquisando"
        print(f"🤖 Assistente:\n{resposta}\n")
        print("-" * 50)

    except Exception as e:
        print(f"\n❌ Ocorreu um erro: {e}\n")